# 05 — Video Inference
**Drone Detection | ITMO Diploma Thesis**

Детекция БПЛА на видео с **любой** дообученной YOLO из проекта (приоритет: **YOLO26s → YOLO12s → YOLO26n → YOLO12n** — см. ячейку с путями). Тот же пайплайн, что и для YOLOv12:
1. Скачать тестовые видео
2. Обработать кадр за кадром
3. Отрисовать bounding boxes
4. Сохранить аннотированное видео
5. Подсчитать реальный FPS

In [ ]:
# ── CELL 1: Install & GPU ─────────────────────────────────────────────────────
!uv pip install ultralytics yt-dlp opencv-python-headless
import ultralytics; ultralytics.checks()
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU!'
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 3: Imports & paths ───────────────────────────────────────────────────
import time
import subprocess
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display
from ultralytics import YOLO
from tqdm import tqdm

# Та же схема каталогов, что в 01_dataset_prep … 04_evaluation (веса в weights/, prepared/ на Drive).
# При другом корне Drive замените DRIVE_ROOT, например: Path('/content/drive/MyDrive/DroneDetection').
DRIVE_ROOT  = Path('/content/drive/MyDrive/Colab Notebooks')
WEIGHTS_DIR = DRIVE_ROOT / 'weights'
RESULTS_DIR = DRIVE_ROOT / 'results'
VIDEO_DIR   = DRIVE_ROOT / 'videos'
VIDEO_OUT   = DRIVE_ROOT / 'video_results'

for _d in (RESULTS_DIR, VIDEO_DIR, VIDEO_OUT):
    _d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']

# Приоритет весов для основного прогона (первый существующий файл используется)
YOLO_WEIGHT_PRIORITY = [
    ('YOLO26s', 'yolo26s_drone_best.pt'),
    ('YOLO12s', 'yolo12s_drone_best.pt'),
    ('YOLO26n', 'yolo26n_drone_best.pt'),
    ('YOLO12n', 'yolo12n_drone_best.pt'),
]

def pick_yolo_weights():
    """Return (short label, path) for the first available fine-tuned checkpoint."""
    for label, fname in YOLO_WEIGHT_PRIORITY:
        p = WEIGHTS_DIR / fname
        if p.exists():
            return label, p
    raise FileNotFoundError(
        f'No YOLO weights in {WEIGHTS_DIR}. Train 02_yolov12_train.ipynb or 02_yolo26_train.ipynb first.'
    )

# BGR colors for OpenCV
CLASS_BGR = {
    'DRONE':      (0,   0,   255),   # Red
    'AIRPLANE':   (255, 0,   0  ),   # Blue
    'HELICOPTER': (0,   255, 0  ),   # Green
    'BIRD':       (0,   255, 255),   # Yellow
}
print('Ready.')

## 1. Скачать тестовые видео

In [ ]:
# ── CELL 4: Download test videos from YouTube ─────────────────────────────────
# Список тестовых роликов (YouTube / Shorts). Добавьте свои пары (url, имя_файла.mp4).
TEST_VIDEOS = [
    ('https://www.youtube.com/shorts/AdinTXYw9Q0', 'drone_flight_1.mp4'),
    ('https://www.youtube.com/shorts/l1tBJG1_lBw', 'drone_flight_2.mp4'),
]


def download_video(url: str, output_path: Path, max_height: int = 720) -> bool:
    """Download a YouTube video using yt-dlp.

    Args:
        url: YouTube video URL (watch, shorts, youtu.be).
        output_path: Desired output path (.mp4); stem used for yt-dlp template.
        max_height: Maximum video height in pixels.

    Returns:
        True if download succeeded.
    """
    dest_dir = output_path.parent
    stem = output_path.stem
    # Шаблон %(ext)s нужен для merge video+audio; итог — <stem>.mp4 при --merge-output-format mp4
    fmt = (
        f'bestvideo[height<={max_height}][ext=mp4]+bestaudio[ext=m4a]/'
        f'best[height<={max_height}][ext=mp4]/best[height<={max_height}]'
    )
    cmd = [
        'yt-dlp',
        '-f', fmt,
        '--merge-output-format', 'mp4',
        '--no-playlist',
        '-o', str(dest_dir / f'{stem}.%(ext)s'),
        url,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        final = dest_dir / f'{stem}.mp4'
        print(f'Downloaded → {final}')
        return True
    err = (result.stderr or '') + (result.stdout or '')
    print(f'Download failed ({url}):\n{err[-800:]}')
    return False


for url, fname in TEST_VIDEOS:
    download_video(url, VIDEO_DIR / fname)

# Либо положите .mp4 вручную в VIDEO_DIR (Colab sidebar / Drive)
print(f'\nVideo directory: {VIDEO_DIR}')
existing = sorted(VIDEO_DIR.glob('*.mp4')) + sorted(VIDEO_DIR.glob('*.avi'))
print(f'Found {len(existing)} video file(s)')
for v in existing:
    print(f'  {v.name}')

## 2. Инференс на видео

In [ ]:
# ── CELL 5: Video inference function ─────────────────────────────────────────
def process_video(
    video_path: Path,
    output_path: Path,
    model: YOLO,
    conf: float = 0.35,
    iou: float = 0.45,
    max_frames: int = 0,
) -> dict:
    """Run YOLO inference on a video file and save annotated output.

    Args:
        video_path: Input video path.
        output_path: Output annotated video path.
        model: Loaded YOLO model.
        conf: Confidence threshold for detections.
        iou: IoU threshold for NMS.
        max_frames: Max frames to process (0 = all).

    Returns:
        Dict with stats: fps, total_frames, detections_per_class.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {video_path}')

    orig_fps    = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_in    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    to_process  = total_in if max_frames == 0 else min(max_frames, total_in)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, orig_fps, (frame_w, frame_h))

    det_counts: dict[str, int] = {n: 0 for n in CLASS_NAMES}
    frame_times = []
    frame_idx = 0

    pbar = tqdm(total=to_process, desc=f'Processing {video_path.name}')

    while frame_idx < to_process:
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.perf_counter()
        results = model.predict(frame, conf=conf, iou=iou, device=0, verbose=False)
        t1 = time.perf_counter()
        frame_times.append(t1 - t0)

        annotated = frame.copy()
        result = results[0]

        for box, score, cls_id in zip(
                result.boxes.xyxy.cpu().numpy(),
                result.boxes.conf.cpu().numpy(),
                result.boxes.cls.cpu().numpy().astype(int)):
            x1, y1, x2, y2 = map(int, box)
            cls_name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'cls{cls_id}'
            color = CLASS_BGR.get(cls_name, (255, 255, 255))
            det_counts[cls_name] = det_counts.get(cls_name, 0) + 1

            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            label = f'{cls_name} {score:.2f}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(annotated, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
            cv2.putText(annotated, label, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1, cv2.LINE_AA)

        # FPS counter (bottom-left)
        current_fps = 1.0 / np.mean(frame_times[-10:]) if frame_times else 0
        cv2.putText(annotated, f'FPS: {current_fps:.1f}',
                    (10, frame_h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        writer.write(annotated)
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()

    avg_fps = 1.0 / np.mean(frame_times) if frame_times else 0
    print(f'\nOutput: {output_path}')
    print(f'Frames processed: {frame_idx}')
    print(f'Average FPS: {avg_fps:.1f}')
    print(f'Detections: {det_counts}')

    return {'fps': round(avg_fps, 1), 'frames': frame_idx, 'detections': det_counts}


print('Function ready.')

In [ ]:
# ── CELL 6: Load model and run inference ──────────────────────────────────────
PRIMARY_LABEL, PRIMARY_PATH = pick_yolo_weights()
model_primary = YOLO(str(PRIMARY_PATH))
print(f'Primary model: {PRIMARY_LABEL} ← {PRIMARY_PATH.name}')

# Дополнительно: все доступные чекпоинты для сравнения скорости (ячейка 7)
model_by_label: dict[str, YOLO] = {PRIMARY_LABEL: model_primary}
for lab, fn in [('YOLO12n', 'yolo12n_drone_best.pt'), ('YOLO12s', 'yolo12s_drone_best.pt'),
                ('YOLO26n', 'yolo26n_drone_best.pt'), ('YOLO26s', 'yolo26s_drone_best.pt')]:
    p = WEIGHTS_DIR / fn
    if p.exists() and lab not in model_by_label:
        model_by_label[lab] = YOLO(str(p))

video_files = list(VIDEO_DIR.glob('*.mp4')) + list(VIDEO_DIR.glob('*.avi'))

if not video_files:
    print('No videos found. Add .mp4 files to:', VIDEO_DIR)
else:
    video_stats: dict = {}
    tag = PRIMARY_LABEL.lower().replace('-', '')
    for vf in video_files:
        print(f'\nProcessing: {vf.name}')
        out_p = VIDEO_OUT / f'{vf.stem}_{tag}.mp4'
        stats_p = process_video(vf, out_p, model_primary, conf=0.35, max_frames=300)
        video_stats[vf.name] = {PRIMARY_LABEL: stats_p}

    print('\nAll videos processed!')

In [ ]:
# ── CELL 7: Compare available YOLO checkpoints on one video ─────────────────
if video_files:
    test_vid = video_files[0]
    print(f'Speed comparison on: {test_vid.name}\n')

    compare_labels = ['YOLO26n', 'YOLO26s', 'YOLO12n', 'YOLO12s']
    results_lines = []
    for lab in compare_labels:
        if lab not in model_by_label:
            continue
        m = model_by_label[lab]
        stem = lab.lower().replace('-', '')
        st = process_video(
            test_vid,
            VIDEO_OUT / f'{test_vid.stem}_{stem}_bench.mp4',
            m,
            conf=0.35,
            max_frames=200,
        )
        results_lines.append(f'{lab}: {st["fps"]} FPS')

    print('\n─── Speed Comparison ───')
    for line in results_lines:
        print(line)

In [ ]:
# ── CELL 8: Show sample frames from output video ──────────────────────────────
def extract_frames(video_path: Path, n_frames: int = 6) -> list[np.ndarray]:
    """Extract evenly spaced frames from a video.

    Args:
        video_path: Path to video file.
        n_frames: Number of frames to extract.

    Returns:
        List of BGR frames as numpy arrays.
    """
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    cap.release()
    return frames


if video_files:
    _tag = PRIMARY_LABEL.lower().replace('-', '')
    out_video = VIDEO_OUT / f'{video_files[0].stem}_{_tag}.mp4'
    if out_video.exists():
        frames = extract_frames(out_video, n_frames=6)

        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        fig.suptitle(f'{PRIMARY_LABEL} — Sample Frames from {out_video.name}',
                     fontsize=12, fontweight='bold')

        for ax, frame in zip(axes, frames):
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.axis('off')

        for ax in axes[len(frames):]:
            ax.set_visible(False)

        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'video_frames_sample.png', dpi=120)
        plt.show()

In [ ]:
# ── CELL 9: YOLO built-in video inference (alternative, easiest) ──────────────
# Ultralytics has built-in video processing — simpler but less control
if video_files:
    _builtin_name = f'{PRIMARY_LABEL.lower().replace("-", "")}_builtin'
    result = model_primary.predict(
        source=str(video_files[0]),
        conf=0.35,
        iou=0.45,
        device=0,
        save=True,
        project=str(VIDEO_OUT),
        name=_builtin_name,
        stream=True,
        max_det=50,
    )
    for r in tqdm(result, desc='Frames'):
        pass
    print(f'Built-in inference complete → {VIDEO_OUT}/{_builtin_name}/')

In [ ]:
# ── CELL 10: Summary ──────────────────────────────────────────────────────────
import json

video_summary = {
    'model_used': PRIMARY_LABEL,
    'weights_file': PRIMARY_PATH.name,
    'conf_threshold': 0.35,
    'videos_processed': len(video_files) if video_files else 0,
    'output_dir': str(VIDEO_OUT),
}

with open(RESULTS_DIR / 'video_inference_summary.json', 'w') as f:
    json.dump(video_summary, f, indent=2)

print('=' * 55)
print('VIDEO INFERENCE COMPLETE')
print('=' * 55)
print(f'Annotated videos saved to: {VIDEO_OUT}')
print(f'Sample frames saved to:    {RESULTS_DIR}/video_frames_sample.png')
print('\nProject complete! Ready for thesis writing.')